In [121]:
import pycutest

probs = pycutest.find_problems()
print(sorted(probs)[:700])
print(len(probs))

['10FOLDTR', '10FOLDTRLS', '3PK', 'A0ENDNDL', 'A0ENINDL', 'A0ENSNDL', 'A0ESDNDL', 'A0ESINDL', 'A0ESSNDL', 'A0NNDNDL', 'A0NNDNIL', 'A0NNDNSL', 'A0NNSNSL', 'A0NSDSDL', 'A0NSDSDS', 'A0NSDSIL', 'A0NSDSSL', 'A0NSSSSL', 'A2ENDNDL', 'A2ENINDL', 'A2ENSNDL', 'A2ESDNDL', 'A2ESINDL', 'A2ESSNDL', 'A2NNDNDL', 'A2NNDNIL', 'A2NNDNSL', 'A2NNSNSL', 'A2NSDSDL', 'A2NSDSIL', 'A2NSDSSL', 'A2NSSSSL', 'A4X12', 'A5ENDNDL', 'A5ENINDL', 'A5ENSNDL', 'A5ESDNDL', 'A5ESINDL', 'A5ESSNDL', 'A5NNDNDL', 'A5NNDNIL', 'A5NNDNSL', 'A5NNSNSL', 'A5NSDSDL', 'A5NSDSDM', 'A5NSDSIL', 'A5NSDSSL', 'A5NSSNSM', 'A5NSSSSL', 'ACOPP118', 'ACOPP14', 'ACOPP30', 'ACOPP300', 'ACOPP57', 'ACOPR118', 'ACOPR14', 'ACOPR30', 'ACOPR300', 'ACOPR57', 'AGG', 'AIRCRFTA', 'AIRCRFTB', 'AIRPORT', 'AKIVA', 'ALJAZZAF', 'ALLINIT', 'ALLINITA', 'ALLINITC', 'ALLINITU', 'ALLINQP', 'ALSOTAME', 'ANTWERP', 'ARGAUSS', 'ARGLALE', 'ARGLBLE', 'ARGLCLE', 'ARGLINA', 'ARGLINB', 'ARGLINC', 'ARGTRIG', 'ARGTRIGLS', 'ARTIF', 'ARWHDNE', 'ARWHEAD', 'AUG2D', 'AUG2DC', 'AUG2DCQ

In [122]:
#HS71

import pycutest
from cyipopt import Problem
import numpy as np

def test_pycutest_problem(problem_name):
    print(f"Testing problem: {problem_name}")
    problem = pycutest.import_problem(problem_name)

    class CUTEstProblem:
        def __init__(self, problem):
            self.problem = problem

        def objective(self, x):
            # Objective function
            return self.problem.obj(x)

        def gradient(self, x):
            # Gradient of the objective
            return self.problem.grad(x)

        def constraints(self, x):
            # Constraint values
            return self.problem.cons(x) if self.problem.m > 0 else np.array([])

        def jacobian(self, x):
            # Jacobian of constraints
            return np.concatenate((np.prod(x) / x, 2*x))

        def hessianstructure(self):
            # Hessian structure (lower triangular indices)
            return np.nonzero(np.tril(np.ones((self.problem.n, self.problem.n))))

        def hessian(self, x, lagrange, obj_factor):
            # Hessian of the Lagrangian
            H = obj_factor*np.array((
                (2*x[3], 0, 0, 0),
                (x[3],   0, 0, 0),
                (x[3],   0, 0, 0),
                (2*x[0]+x[1]+x[2], x[0], x[0], 0)))

            H += lagrange[0]*np.array((
                    (0, 0, 0, 0),
                    (x[2]*x[3], 0, 0, 0),
                    (x[1]*x[3], x[0]*x[3], 0, 0),
                    (x[1]*x[2], x[0]*x[2], x[0]*x[1], 0)))
    
            H += lagrange[1]*2*np.eye(4)
    
            row, col = self.hessianstructure()
    
            return H[row, col]

        def intermediate(
            self,
            alg_mod,
            iter_count,
            obj_value,
            inf_pr,
            inf_du,
            mu,
            d_norm,
            regularization_size,
            alpha_du,
            alpha_pr,
            ls_trials
        ):
            # Intermediate callback
            print(f"Iteration {iter_count}: Objective value = {obj_value}")

    cutest_problem = CUTEstProblem(problem)

    # Variable bounds
    lb = problem.bl
    ub = problem.bu

    # Constraint bounds
    cl = problem.cl if problem.m > 0 else []
    cu = problem.cu if problem.m > 0 else []
    print(cl)
    print(cu)

    # Initial guess
    x0 = problem.x0
    print(problem.obj(x0))
    print(problem.grad(x0))
    print(problem.cons(x0))
    print(problem.jprod(x0, transpose=False))

    # Create Ipopt problem instance
    nlp = Problem(
        n=problem.n,
        m=problem.m,
        problem_obj=cutest_problem,
        lb=lb,
        ub=ub,
        cl=cl,
        cu=cu
    )

    # Solver options
    nlp.add_option('mu_strategy', 'adaptive')
    nlp.add_option('tol', 1e-7)
    nlp.add_option('print_level', 5)

    # Solve the problem
    x, info = nlp.solve(x0)

    # Results
    print(f"Solution of the primal variables: x = {x}")
    print(f"Solution of the dual variables: lambda = {info['mult_g']}")
    print(f"Objective value: {info['obj_val']}")

# Test the HS71 problem
test_pycutest_problem("HS71")

Testing problem: HS71
[0. 0.]
[1.e+20 0.e+00]
16.0
[12.  1.  2. 11.]
[ 0. 12.]
[0. 0.]
Iteration 0: Objective value = 16.109692919198878
Iteration 1: Objective value = 17.263330577871336
Iteration 2: Objective value = 17.829620922173714
Iteration 3: Objective value = 17.353145550775487
Iteration 4: Objective value = 16.95081187222606
Iteration 5: Objective value = 17.00282819438091
Iteration 6: Objective value = 17.013923998570412
Iteration 7: Objective value = 17.014017058215963
Iteration 8: Objective value = 17.01401727277449
Solution of the primal variables: x = [0.99999999 4.74299964 3.82114998 1.37940831]
Solution of the dual variables: lambda = [-0.55229366  0.16146857]
Objective value: 17.01401727277449
This is Ipopt version 3.14.17, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:        4
Number of nonzeros in inequality constraint Jacobian.:        4
Number of nonzeros in Lagrangian Hessian.............:       10

Total number of

In [119]:
# Other HS Examples

import pycutest
from cyipopt import Problem
import numpy as np

def test_pycutest_problem(problem_name):
    print(f"Testing problem: {problem_name}")
    problem = pycutest.import_problem(problem_name)

    class CUTEstProblem:
        def __init__(self, problem):
            self.problem = problem

        def objective(self, x):
            # Objective function
            return self.problem.obj(x)

        def gradient(self, x):
            # Gradient of the objective
            return self.problem.grad(x)

        def constraints(self, x):
            # Constraint values
            if problem_name == "HS25":
                return np.array([x[0] ** 2 + x[1] ** 2 - 1])
            elif problem_name == "HS38":
                return np.array([x[0] * x[1] - 1, x[0] ** 2 + x[1] ** 2 - 4])
            elif problem_name == "HS52":
                return np.array([
                    x[0] + 3 * x[1],
                    x[2] + x[3] - 2 * x[4],
                    x[1] - x[4]
                ])
            elif problem_name == "HS80":
                return np.array([
                    x[0]**2 + x[1]**2 + x[2]**2 + x[3]**2 + x[4]**2 - 10,
                    x[1] * x[2] - 5 * x[3] * x[4],
                    x[0]**3 + x[1]**3 + 1
                ])
            else:
                return self.problem.cons(x) if self.problem.m > 0 else np.array([])

        def jacobian(self, x):
            # Jacobian of constraints
            if problem_name == "HS25":
                return np.array([[2 * x[0], 2 * x[1]]])
            elif problem_name == "HS38":
                return np.array([
                    [x[1], x[0]],
                    [2 * x[0], 2 * x[1]]
                ])
            elif problem_name == "HS52":
                return np.array([
                    [1, 3, 0, 0, 0],   # Gradient of the first constraint
                    [0, 0, 1, 1, -2],  # Gradient of the second constraint
                    [0, 1, 0, 0, -1]   # Gradient of the third constraint
                ])
            elif problem_name == "HS80":
                return np.array([
                    [2 * x[0], 2 * x[1], 2 * x[2], 2 * x[3], 2 * x[4]],   # First constraint
                    [0, x[2], x[1], -5 * x[4], -5 * x[3]],                # Second constraint
                    [3 * x[0]**2, 3 * x[1]**2, 0, 0, 0]                   # Third constraint
                ])
            else:
                return self.problem.jprod(x, transpose=False)

        def hessianstructure(self):
            # Hessian structure (lower triangular indices)
            return np.nonzero(np.tril(np.ones((self.problem.n, self.problem.n))))

        def hessian(self, x, lagrange, obj_factor):
            # Hessian of the Lagrangian
            hess = obj_factor * self.problem.hess(x)
            if self.problem.m > 0:
                for i in range(self.problem.m):
                    hess += lagrange[i] * self.problem.hess(x, con_idx=i)
            row, col = self.hessianstructure()
            return hess[row, col]

        def intermediate(
            self,
            alg_mod,
            iter_count,
            obj_value,
            inf_pr,
            inf_du,
            mu,
            d_norm,
            regularization_size,
            alpha_du,
            alpha_pr,
            ls_trials
        ):
            print(f"Iteration {iter_count}: Objective value = {obj_value}")

    cutest_problem = CUTEstProblem(problem)

    # Variable bounds
    lb = problem.bl
    ub = problem.bu

    # Constraint bounds
    cl = problem.cl if problem.m > 0 else []
    cu = problem.cu if problem.m > 0 else []
    print(cl)
    print(cu)

    # Initial guess
    x0 = problem.x0

    # Create Ipopt problem instance
    nlp = Problem(
        n=problem.n,
        m=problem.m,
        problem_obj=cutest_problem,
        lb=lb,
        ub=ub,
        cl=cl,
        cu=cu
    )

    # Solver options
    nlp.add_option('mu_strategy', 'adaptive')
    nlp.add_option('tol', 1e-7)
    nlp.add_option('print_level', 5)

    # Solve the problem
    x, info = nlp.solve(x0)

    # Results
    print(f"Solution of the primal variables: x = {x}")
    print(f"Solution of the dual variables: lambda = {info['mult_g']}")
    print(f"Objective value: {info['obj_val']}")
    print("=" * 50)

# Test the specified problems
problems_to_test = ["HS25", "HS38", "HS52","HS80"]
for prob_name in problems_to_test:
    test_pycutest_problem(prob_name)

Testing problem: HS25
[]
[]
Iteration 0: Objective value = 32.83499999973319
Iteration 1: Objective value = 32.83499999973313
This is Ipopt version 3.14.17, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        6

Total number of variables............................:        3
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        3
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   in

In [120]:
#Comparison to barrier methods (barrier methods do not work)

import numpy as np
from scipy.optimize import minimize
import pycutest


def barrier_objective(x, mu, objective, constraints):
    """Barrier method objective with logarithmic penalty for constraints."""
    obj_val = objective(x)
    barrier_term = 0
    for g in constraints(x):
        if g >= 0:
            return np.inf  # Return infinity if the constraint is violated
        barrier_term -= mu * np.log(-g)
    return obj_val + barrier_term


def barrier_method(problem_name, objective, gradient, constraints, x0, bounds, mu_start=1.0, tol=1e-6, max_iter=100):
    """Implementation of the barrier method."""
    x = x0
    mu = mu_start

    print(f"Testing problem: {problem_name}")

    for iter in range(max_iter):
        # Define barrier objective
        barrier_obj = lambda x: barrier_objective(x, mu, objective, constraints)
        barrier_grad = gradient  # Use the provided gradient

        # Solve unconstrained problem with the barrier term
        res = minimize(barrier_obj, x, jac=barrier_grad, bounds=bounds, method='L-BFGS-B')

        # Check convergence
        if res.success and np.linalg.norm(res.jac) < tol:
            print(f"Converged at iteration {iter} with objective value: {res.fun}")
            return res.x, res.fun

        # Update x and reduce the barrier parameter
        x = res.x
        mu /= 10

    print(f"Barrier method did not converge for {problem_name}.")
    return x, None


def test_barrier_method(problem_name):
    problem = pycutest.import_problem(problem_name)

    # Define objective, gradient, and constraints for each problem
    def objective(x):
        return problem.obj(x)

    def gradient(x):
        return problem.grad(x)

    def constraints(x):
        if problem_name == "HS25":
            return np.array([x[0] ** 2 + x[1] ** 2 - 1])
        elif problem_name == "HS38":
            return np.array([x[0] * x[1] - 1, x[0] ** 2 + x[1] ** 2 - 4])
        elif problem_name == "HS52":
            return np.array([
                x[0] + 3 * x[1],
                x[2] + x[3] - 2 * x[4],
                x[1] - x[4]
            ])
        elif problem_name == "HS80":
            return np.array([
                x[0]**2 + x[1]**2 + x[2]**2 + x[3]**2 + x[4]**2 - 10,
                x[1] * x[2] - 5 * x[3] * x[4],
                x[0]**3 + x[1]**3 + 1
            ])
        else:
            return problem.cons(x) if problem.m > 0 else np.array([])

    # Set bounds and initial guess
    lb = problem.bl
    ub = problem.bu
    x0 = problem.x0
    bounds = [(lb[i], ub[i]) for i in range(len(lb))]

    # Run the barrier method
    solution, obj_value = barrier_method(problem_name, objective, gradient, constraints, x0, bounds)

    # Print results
    if obj_value is not None:
        print(f"Solution for {problem_name}: {solution}")
        print(f"Objective value: {obj_value}")
    else:
        print(f"Barrier method failed for {problem_name}.")

    print("=" * 50)


# Test the specified problems
problems_to_test = ["HS25", "HS38", "HS52", "HS80"]
for prob_name in problems_to_test:
    test_barrier_method(prob_name)

Testing problem: HS25
Converged at iteration 0 with objective value: inf
Solution for HS25: [100.   12.5   3. ]
Objective value: inf
Testing problem: HS38
Barrier method did not converge for HS38.
Barrier method failed for HS38.
Testing problem: HS52
Converged at iteration 0 with objective value: inf
Solution for HS52: [0.30303034 1.21212135 0.78787894 1.00000011 1.00000011]
Objective value: inf
Testing problem: HS80
Barrier method did not converge for HS80.
Barrier method failed for HS80.
